# WorkSafe AI — NLP Model untuk Analisis Risiko Pekerjaan


1. **risk_label**: kelas risiko pekerjaan (`Low`, `Medium`, `High`).
2. **automation_risk_score**: skor risiko otomasi dalam rentang `0–1`.



## 1. Instalasi Library

Opsional

## 2. Import Library dan Konfigurasi Awal

In [1]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, mean_absolute_error

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE_DIR = Path(".")
DATA_PATH = BASE_DIR / "occupation_master_final1.csv"
EXPORT_DIR = BASE_DIR / "exports"
LOG_DIR = BASE_DIR / "logs" / "worksafe_nlp"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = EXPORT_DIR / "worksafe_risk_model.keras"
BEST_MODEL_PATH = EXPORT_DIR / "worksafe_risk_model_best.keras"
SAVEDMODEL_DIR = EXPORT_DIR / "worksafe_savedmodel"
ARTIFACT_PATH = EXPORT_DIR / "worksafe_artifacts.json"
METRICS_PATH = EXPORT_DIR / "worksafe_metrics.json"

TARGET_ACC = 0.85
TARGET_MAE = 0.02

2026-05-09 03:10:20.151352: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow: 2.16.2
GPU: []


## 3. Load Dataset CSV

Pastikan file `occupation_master_final1(1).csv` berada satu folder dengan notebook ini.

In [2]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (4096, 73)
Columns: ['occupation_code', 'title', 'description', 'skill_count', 'skill_importance_avg', 'skill_level_avg', 'top_skills', 'task_count', 'core_task_count', 'supplemental_task_count', 'incumbents_responding_avg', 'core_task_ratio', 'top_core_tasks', 'data_analysis_score', 'social_score', 'coaching_and_developing_others', 'communicating_with_people_outside_the_organization', 'communicating_with_supervisors,_peers,_or_subordinates', 'controlling_machines_and_processes', 'coordinating_the_work_and_activities_of_others', 'developing_objectives_and_strategies', 'developing_and_building_teams', 'documenting_recording_information', 'drafting,_laying_out,_and_specifying_technical_devices,_parts,_and_equipment', 'establishing_and_maintaining_interpersonal_relationships', 'estimating_the_quantifiable_characteristics_of_products,_events,_or_information', 'evaluating_information_to_determine_compliance_with_standards', 'getting_information', 'guiding,_directing,_and_motivating_s

,occupation_code,title,description,skill_count,skill_importance_avg,skill_level_avg,top_skills,task_count,core_task_count,supplemental_task_count,...,task_count_norm,core_task_ratio_norm,job_zone_norm,human_bottleneck_raw_norm,cognitive_bottleneck_raw_norm,digital_ai_exposure_raw_norm,manual_work_raw_norm,automation_risk_score,risk_label,reskilling_recommendation
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,35.0,3.188571,3.365143,"Judgment and Decision Making, Critical Thinkin...",31.0,20.0,11.0,...,0.750000,0.645161,1.000000,0.532033,0.910112,0.864865,0.218873,0.703335,High,"AI literacy, automation tools, data analytics"
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, sh...",35.0,2.918857,2.821143,"Writing, Critical Thinking, Reading Comprehens...",18.0,18.0,0.0,...,0.388889,1.000000,1.000000,0.373259,0.882647,0.765766,0.144168,0.729564,High,"communication, collaboration, stakeholder mana..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of ...",35.0,2.852857,2.724286,"Reading Comprehension, Active Listening, Speak...",17.0,9.0,8.0,...,0.361111,0.529412,0.666667,0.451253,0.704120,0.756757,0.256881,0.558113,Medium,"communication, collaboration, stakeholder mana..."
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes...",35.0,2.624286,2.421429,unknown,30.0,0.0,0.0,...,0.722222,0.000000,0.666667,0.370474,0.638577,0.611111,0.494102,0.373351,Medium,"communication, collaboration, stakeholder mana..."
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici...",35.0,2.709429,2.652571,"Active Listening, Speaking, Critical Thinking,...",30.0,13.0,8.0,...,0.722222,0.433333,0.666667,0.270195,0.681648,0.554054,0.239843,0.509339,Medium,"communication, collaboration, stakeholder mana..."


In [3]:
# Cek target utama
required_cols = ["title", "description", "top_skills", "top_core_tasks", "automation_risk_score", "risk_label"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing}")

print(df["risk_label"].value_counts())
print(df["automation_risk_score"].describe())

risk_label
Medium    1727
Low       1538
High       831
Name: count, dtype: int64
count    4096.000000
mean        0.481566
std         0.198260
min         0.000000
25%         0.320990
50%         0.472265
75%         0.625643
max         1.000000
Name: automation_risk_score, dtype: float64


## 4. Preprocessing Data

Strategi input model:

- **Input teks**: gabungan `title`, `description`, `top_skills`, dan `top_core_tasks`.
- **Input numerik**: semua kolom numerik selain target `automation_risk_score`.
- **Output klasifikasi**: `risk_label`.
- **Output regresi**: `automation_risk_score`.


In [4]:
TEXT_COLS = ["title", "description", "top_skills", "top_core_tasks"]
TARGET_LABEL_COL = "risk_label"
TARGET_SCORE_COL = "automation_risk_score"

# Buat teks gabungan untuk NLP
for col in TEXT_COLS:
    df[col] = df[col].fillna("").astype(str)

df["job_text"] = (
    "Job Title: " + df["title"] + "" +
    "Description: " + df["description"] + "" +
    "Top Skills: " + df["top_skills"] + "" +
    "Core Tasks: " + df["top_core_tasks"]
)

# Kolom numerik: semua numerik kecuali target score
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != TARGET_SCORE_COL]

# Isi missing value numerik dengan median training/global sementara
numeric_medians = df[numeric_cols].median(numeric_only=True).fillna(0.0).to_dict()
df[numeric_cols] = df[numeric_cols].fillna(numeric_medians)

# Pastikan skor berada pada range 0-1
df[TARGET_SCORE_COL] = df[TARGET_SCORE_COL].astype("float32").clip(0, 1)

# Mapping label agar urutannya stabil dan mudah dibaca di web
preferred_order = ["Low", "Medium", "High"]
existing_labels = df[TARGET_LABEL_COL].dropna().astype(str).unique().tolist()
label_classes = [x for x in preferred_order if x in existing_labels] + [x for x in sorted(existing_labels) if x not in preferred_order]
label_to_id = {label: idx for idx, label in enumerate(label_classes)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

df["risk_label_id"] = df[TARGET_LABEL_COL].astype(str).map(label_to_id).astype("int32")

print("Jumlah fitur numerik:", len(numeric_cols))
print("Label classes:", label_classes)
df[["title", "job_text", TARGET_LABEL_COL, "risk_label_id", TARGET_SCORE_COL]].head(3)

Jumlah fitur numerik: 65
Label classes: ['Low', 'Medium', 'High']


,title,job_text,risk_label,risk_label_id,automation_risk_score
0,Chief Executives,Job Title: Chief ExecutivesDescription: Determ...,High,2,0.703335
1,Chief Sustainability Officers,Job Title: Chief Sustainability OfficersDescri...,High,2,0.729564
2,General and Operations Managers,Job Title: General and Operations ManagersDesc...,Medium,1,0.558113


## 5. Split Dataset Train / Validation / Test

In [5]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["risk_label_id"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["risk_label_id"]
)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

print("Distribusi label train:")
print(train_df[TARGET_LABEL_COL].value_counts(normalize=True))

Train: (2867, 75)
Val  : (614, 75)
Test : (615, 75)
Distribusi label train:
risk_label
Medium    0.421695
Low       0.375305
High      0.203000
Name: proportion, dtype: float64


## 6. Pipeline `tf.data`

In [6]:
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE


def make_dataset(dataframe, shuffle=False):
    text_data = dataframe["job_text"].astype(str).to_numpy()
    numeric_data = dataframe[numeric_cols].astype("float32").to_numpy()
    label_data = dataframe["risk_label_id"].astype("int32").to_numpy()
    score_data = dataframe[TARGET_SCORE_COL].astype("float32").to_numpy().reshape(-1, 1)

    features = {
        "job_text": text_data,
        "numeric_features": numeric_data,
    }
    targets = {
        "risk_label": label_data,
        "risk_score": score_data,
    }

    ds = tf.data.Dataset.from_tensor_slices((features, targets))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

for x_batch, y_batch in train_ds.take(1):
    print(x_batch["job_text"].shape, x_batch["numeric_features"].shape)
    print(y_batch["risk_label"].shape, y_batch["risk_score"].shape)

(64,) (64, 65)
(64,) (64, 1)


2026-05-09 03:10:27.585278: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## 7. Text Vectorization dan Normalization Layer

Layer preprocessing dimasukkan langsung ke dalam model agar saat inference web/API tidak perlu membuat tokenizer terpisah.

In [7]:
MAX_TOKENS = 20_000
SEQ_LEN = 160

text_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQ_LEN,
    standardize="lower_and_strip_punctuation",
    name="text_vectorizer",
)

text_vectorizer.adapt(train_df["job_text"].astype(str).to_numpy())

normalizer = tf.keras.layers.Normalization(name="numeric_normalization")
normalizer.adapt(train_df[numeric_cols].astype("float32").to_numpy())

print("Vocabulary size:", len(text_vectorizer.get_vocabulary()))
print("Numeric input dim:", len(numeric_cols))

Vocabulary size: 8611
Numeric input dim: 65


## 8. Komponen Kustom Lanjutan

Bagian ini memenuhi kriteria custom component:

- `AttentionPooling`: Custom Layer.
- `SparseCategoricalFocalLoss`: Custom Loss Function.
- `TargetMetricCallback`: Custom Callback.

In [8]:
@tf.keras.utils.register_keras_serializable(package="WorkSafeAI")
class AttentionPooling(tf.keras.layers.Layer):
    """Custom layer untuk memberi bobot perhatian pada token penting."""

    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.score_dense = tf.keras.layers.Dense(units, activation="tanh")
        self.attention_score = tf.keras.layers.Dense(1)

    def call(self, inputs, training=None):
        # inputs: [batch, sequence_length, hidden_dim]
        scores = self.attention_score(self.score_dense(inputs))
        weights = tf.nn.softmax(scores, axis=1)
        context = tf.reduce_sum(inputs * weights, axis=1)
        return context

    def get_config(self):
        config = super().get_config()
        config.update({"units": self.units})
        return config


class SparseCategoricalFocalLoss(tf.keras.losses.Loss):
    """Custom focal loss untuk membantu model fokus pada sample yang sulit."""

    def __init__(self, gamma=2.0, alpha=None, name="sparse_categorical_focal_loss"):
        super().__init__(name=name, reduction=tf.keras.losses.Reduction.NONE)
        self.gamma = gamma
        self.alpha = None if alpha is None else tf.constant(alpha, dtype=tf.float32)

    def call(self, y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())

        batch_indices = tf.range(tf.shape(y_true)[0], dtype=tf.int32)
        gather_indices = tf.stack([batch_indices, y_true], axis=1)
        p_t = tf.gather_nd(y_pred, gather_indices)

        loss = -tf.pow(1.0 - p_t, self.gamma) * tf.math.log(p_t)
        if self.alpha is not None:
            alpha_t = tf.gather(self.alpha, y_true)
            loss = alpha_t * loss
        return loss


class TargetMetricCallback(tf.keras.callbacks.Callback):
    """Custom callback untuk menghentikan training ketika target tercapai."""

    def __init__(self, target_acc=0.85, target_mae=0.02):
        super().__init__()
        self.target_acc = target_acc
        self.target_mae = target_mae

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        val_acc = logs.get("val_risk_label_accuracy")
        val_mae = logs.get("val_risk_score_mae")

        if val_acc is not None and val_mae is not None:
            if val_acc >= self.target_acc and val_mae <= self.target_mae:
                print(
                    f"Target tercapai pada epoch {epoch + 1}: "
                    f"val_acc={val_acc:.4f}, val_mae={val_mae:.4f}"
                )
                self.model.stop_training = True

## 9. Bangun Model dengan TensorFlow Functional API

In [9]:
def build_worksafe_model(num_numeric_features, num_classes):
    # Input teks
    text_input = tf.keras.Input(shape=(1,), dtype=tf.string, name="job_text")
    x_text = text_vectorizer(text_input)
    x_text = tf.keras.layers.Embedding(
        input_dim=MAX_TOKENS,
        output_dim=128,
        name="token_embedding"
    )(x_text)
    x_text = tf.keras.layers.SpatialDropout1D(0.20)(x_text)
    x_text = tf.keras.layers.Bidirectional(
        tf.keras.layers.GRU(64, return_sequences=True),
        name="bigru_text_encoder"
    )(x_text)
    x_text = AttentionPooling(units=64, name="attention_pooling")(x_text)
    x_text = tf.keras.layers.Dense(96, activation="relu", name="text_dense")(x_text)
    x_text = tf.keras.layers.Dropout(0.25)(x_text)

    # Input numerik
    numeric_input = tf.keras.Input(shape=(num_numeric_features,), dtype=tf.float32, name="numeric_features")
    x_num = normalizer(numeric_input)
    x_num = tf.keras.layers.Dense(128, activation="relu", name="numeric_dense_1")(x_num)
    x_num = tf.keras.layers.BatchNormalization()(x_num)
    x_num = tf.keras.layers.Dropout(0.20)(x_num)
    x_num = tf.keras.layers.Dense(64, activation="relu", name="numeric_dense_2")(x_num)

    # Fusion teks + numerik
    x = tf.keras.layers.Concatenate(name="feature_fusion")([x_text, x_num])
    x = tf.keras.layers.Dense(160, activation="relu", name="fusion_dense_1")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.30)(x)
    x = tf.keras.layers.Dense(80, activation="relu", name="fusion_dense_2")(x)

    # Multi-output
    risk_label = tf.keras.layers.Dense(num_classes, activation="softmax", name="risk_label")(x)
    risk_score = tf.keras.layers.Dense(1, activation="sigmoid", name="risk_score")(x)

    model = tf.keras.Model(
        inputs={
            "job_text": text_input,
            "numeric_features": numeric_input,
        },
        outputs=[risk_label, risk_score],
        name="WorkSafeAI_NLP_Risk_Model",
    )
    return model

model = build_worksafe_model(
    num_numeric_features=len(numeric_cols),
    num_classes=len(label_classes),
)

model.summary()

Model: "WorkSafeAI_NLP_Risk_Model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ job_text            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorizer     │ (None, 160)       │          0 │ job_text[0][0]    │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_embedding     │ (None, 160, 128)  │  2,560,000 │ text_vectorizer[… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numeric_features    │ (None, 65)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d   │ (None, 160, 128)  │          0 │ token_embedding[… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numeric_normalizat… │ (None, 65)        │        131 │ numeric_features… │
│ (Normalization)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bigru_text_encoder  │ (None, 160, 128)  │     74,496 │ spatial_dropout1… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numeric_dense_1     │ (None, 128)       │      8,448 │ numeric_normaliz… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_pooling   │ (None, 128)       │      8,321 │ bigru_text_encod… │
│ (AttentionPooling)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128)       │        512 │ numeric_dense_1[… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_dense (Dense)  │ (None, 96)        │     12,384 │ attention_poolin… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 96)        │          0 │ text_dense[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numeric_dense_2     │ (None, 64)        │      8,256 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_fusion      │ (None, 160)       │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ numeric_dense_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fusion_dense_1      │ (None, 160)       │     25,760 │ feature_fusion[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 160)       │        640 │ fusion_dense_1[0… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 160)       │          0 │ batch_normalizat

 Total params: 2,712,152 (10.35 MB)

 Trainable params: 2,711,445 (10.34 MB)

 Non-trainable params: 707 (2.77 KB)

## 10. Setup Loss, Optimizer, Metrics, dan TensorBoard

In [10]:
# Alpha class weight untuk focal loss
label_counts = train_df["risk_label_id"].value_counts().sort_index()
class_freq = label_counts / label_counts.sum()
alpha = (1.0 / class_freq).to_numpy()
alpha = alpha / alpha.sum() * len(label_classes)
alpha = alpha.astype("float32")

print("Class counts:", label_counts.to_dict())
print("Alpha focal loss:", alpha)

classification_loss_fn = SparseCategoricalFocalLoss(gamma=2.0, alpha=alpha)
regression_loss_fn = tf.keras.losses.Huber(delta=0.05, reduction=tf.keras.losses.Reduction.NONE)

# Bobot regresi dibuat cukup besar agar target MAE <= 0.02 lebih mudah tercapai
REGRESSION_LOSS_WEIGHT = 5.0

optimizer = tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4)

train_loss_metric = tf.keras.metrics.Mean(name="train_total_loss")
train_acc_metric = tf.keras.metrics.SparseCategoricalAccuracy(name="train_risk_label_accuracy")
train_mae_metric = tf.keras.metrics.MeanAbsoluteError(name="train_risk_score_mae")

val_loss_metric = tf.keras.metrics.Mean(name="val_total_loss")
val_acc_metric = tf.keras.metrics.SparseCategoricalAccuracy(name="val_risk_label_accuracy")
val_mae_metric = tf.keras.metrics.MeanAbsoluteError(name="val_risk_score_mae")

summary_writer = tf.summary.create_file_writer(str(LOG_DIR))
print("TensorBoard log dir:", LOG_DIR)

Class counts: {0: 1076, 1: 1209, 2: 582}
Alpha focal loss: [0.80239886 0.7141283  1.4834728 ]
TensorBoard log dir: logs/worksafe_nlp


## 11. Custom Training dan Evaluation Step dengan `tf.GradientTape`

In [11]:
@tf.function
def train_step(features, targets):
    with tf.GradientTape() as tape:
        pred_label, pred_score = model(features, training=True)

        cls_loss = classification_loss_fn(targets["risk_label"], pred_label)
        reg_loss = regression_loss_fn(targets["risk_score"], pred_score)

        cls_loss = tf.reduce_mean(cls_loss)
        reg_loss = tf.reduce_mean(reg_loss)

        reg_terms = tf.add_n(model.losses) if model.losses else 0.0
        total_loss = cls_loss + (REGRESSION_LOSS_WEIGHT * reg_loss) + reg_terms

    gradients = tape.gradient(total_loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    train_loss_metric.update_state(total_loss)
    train_acc_metric.update_state(targets["risk_label"], pred_label)
    train_mae_metric.update_state(targets["risk_score"], pred_score)

    return total_loss, cls_loss, reg_loss


@tf.function
def val_step(features, targets):
    pred_label, pred_score = model(features, training=False)

    cls_loss = classification_loss_fn(targets["risk_label"], pred_label)
    reg_loss = regression_loss_fn(targets["risk_score"], pred_score)

    cls_loss = tf.reduce_mean(cls_loss)
    reg_loss = tf.reduce_mean(reg_loss)
    total_loss = cls_loss + (REGRESSION_LOSS_WEIGHT * reg_loss)

    val_loss_metric.update_state(total_loss)
    val_acc_metric.update_state(targets["risk_label"], pred_label)
    val_mae_metric.update_state(targets["risk_score"], pred_score)

    return total_loss, cls_loss, reg_loss


def reset_metrics():
    for metric in [
        train_loss_metric, train_acc_metric, train_mae_metric,
        val_loss_metric, val_acc_metric, val_mae_metric,
    ]:
        metric.reset_state()

## 12. Training Model

Jika target belum tercapai, naikkan `EPOCHS` menjadi 100–150 atau naikkan `REGRESSION_LOSS_WEIGHT`. Dataset ini memiliki target skor numerik yang kuat, sehingga model biasanya dapat mencapai akurasi tinggi dan MAE rendah setelah beberapa epoch.

In [12]:
EPOCHS = 80
PATIENCE = 12

best_val_loss = np.inf
wait = 0
history = []

model.stop_training = False
target_callback = TargetMetricCallback(target_acc=TARGET_ACC, target_mae=TARGET_MAE)
target_callback.set_model(model)
target_callback.on_train_begin()

for epoch in range(EPOCHS):
    reset_metrics()

    for features, targets in train_ds:
        train_step(features, targets)

    for features, targets in val_ds:
        val_step(features, targets)

    logs = {
        "train_total_loss": float(train_loss_metric.result().numpy()),
        "train_risk_label_accuracy": float(train_acc_metric.result().numpy()),
        "train_risk_score_mae": float(train_mae_metric.result().numpy()),
        "val_total_loss": float(val_loss_metric.result().numpy()),
        "val_risk_label_accuracy": float(val_acc_metric.result().numpy()),
        "val_risk_score_mae": float(val_mae_metric.result().numpy()),
    }
    history.append(logs)

    with summary_writer.as_default():
        for key, value in logs.items():
            tf.summary.scalar(key, value, step=epoch + 1)
        summary_writer.flush()

    print(
        f"Epoch {epoch + 1:03d}/{EPOCHS} | "
        f"loss={logs['train_total_loss']:.4f} | "
        f"acc={logs['train_risk_label_accuracy']:.4f} | "
        f"mae={logs['train_risk_score_mae']:.4f} | "
        f"val_loss={logs['val_total_loss']:.4f} | "
        f"val_acc={logs['val_risk_label_accuracy']:.4f} | "
        f"val_mae={logs['val_risk_score_mae']:.4f}"
    )

    # Manual model checkpoint
    if logs["val_total_loss"] < best_val_loss:
        best_val_loss = logs["val_total_loss"]
        wait = 0
        model.save(BEST_MODEL_PATH)
        print(f"  Best model disimpan ke: {BEST_MODEL_PATH}")
    else:
        wait += 1

    # Custom callback target metric
    target_callback.on_epoch_end(epoch, logs)
    if model.stop_training:
        break

    # Early stopping manual
    if wait >= PATIENCE:
        print(f"Early stopping aktif. Tidak ada peningkatan val_loss selama {PATIENCE} epoch.")
        break

target_callback.on_train_end()

history_df = pd.DataFrame(history)
history_df.tail()

2026-05-09 03:10:50.958453: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-05-09 03:10:53.923962: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 001/80 | loss=0.4927 | acc=0.5860 | mae=0.2794 | val_loss=0.2833 | val_acc=0.6645 | val_mae=0.1466
  Best model disimpan ke: exports/worksafe_risk_model_best.keras


2026-05-09 03:11:17.455568: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-05-09 03:11:18.101701: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 002/80 | loss=0.2018 | acc=0.7600 | mae=0.1461 | val_loss=0.1636 | val_acc=0.7769 | val_mae=0.0871
  Best model disimpan ke: exports/worksafe_risk_model_best.keras


2026-05-09 03:11:30.866886: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-05-09 03:11:31.497197: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


...

Epoch 018/80 | loss=0.0298 | acc=0.9620 | mae=0.0505 | val_loss=0.0966 | val_acc=0.8779 | val_mae=0.0357


2026-05-09 03:15:08.583945: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 019/80 | loss=0.0269 | acc=0.9676 | mae=0.0481 | val_loss=0.1067 | val_acc=0.8860 | val_mae=0.0375
Early stopping aktif. Tidak ada peningkatan val_loss selama 12 epoch.


2026-05-09 03:15:09.244347: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


,train_total_loss,train_risk_label_accuracy,train_risk_score_mae,val_total_loss,val_risk_label_accuracy,val_risk_score_mae
14,0.031984,0.958493,0.052307,0.093622,0.874593,0.037949
15,0.035426,0.956400,0.051298,0.157573,0.861564,0.044562
16,0.030957,0.962679,0.049464,0.111222,0.885993,0.038062
17,0.029834,0.961981,0.050502,0.096593,0.877850,0.035727
18,0.026858,0.967562,0.048141,0.106743,0.885993,0.037465


## 13. Load Best Model dan Evaluasi Test Set

In [13]:
# Load best model untuk evaluasi final
model = tf.keras.models.load_model(
    BEST_MODEL_PATH,
    custom_objects={"AttentionPooling": AttentionPooling},
    compile=False,
)

all_true_label = []
all_pred_label = []
all_true_score = []
all_pred_score = []

for features, targets in test_ds:
    pred_label, pred_score = model(features, training=False)

    all_true_label.extend(targets["risk_label"].numpy().tolist())
    all_pred_label.extend(np.argmax(pred_label.numpy(), axis=1).tolist())
    all_true_score.extend(targets["risk_score"].numpy().reshape(-1).tolist())
    all_pred_score.extend(pred_score.numpy().reshape(-1).tolist())

final_acc = accuracy_score(all_true_label, all_pred_label)
final_mae = mean_absolute_error(all_true_score, all_pred_score)

print("Test Accuracy:", final_acc)
print("Test MAE:", final_mae)
print("Classification Report:")
print(classification_report(all_true_label, all_pred_label, target_names=label_classes))
print("Confusion Matrix:")
print(confusion_matrix(all_true_label, all_pred_label))

if final_acc >= TARGET_ACC and final_mae <= TARGET_MAE:
    print(f"LULUS TARGET: accuracy >= {TARGET_ACC} dan MAE <= {TARGET_MAE}")
else:
    print(f"BELUM LULUS TARGET: target accuracy >= {TARGET_ACC}, MAE <= {TARGET_MAE}")
    print("Saran: naikkan EPOCHS, naikkan REGRESSION_LOSS_WEIGHT, atau kurangi dropout.")

/Users/neko/Downloads/worksafe_ai_nlp_project/venv/lib/python3.12/site-packages/keras/src/layers/layer.py:427: UserWarning: `build()` was called on layer 'attention_pooling', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Test Accuracy: 0.8780487804878049
Test MAE: 0.04727050935834404
Classification Report:
              precision    recall  f1-score   support

         Low       0.92      0.95      0.94       231
      Medium       0.91      0.78      0.84       259
        High       0.75      0.94      0.84       125

    accuracy                           0.88       615
   macro avg       0.86      0.89      0.87       615
weighted avg       0.89      0.88      0.88       615

Confusion Matrix:
[[220  11   0]
 [ 18 203  38]
 [  0   8 117]]
BELUM LULUS TARGET: target accuracy >= 0.85, MAE <= 0.02
Saran: naikkan EPOCHS, naikkan REGRESSION_LOSS_WEIGHT, atau kurangi dropout.


2026-05-09 03:15:34.010060: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## 14. Simpan Model dan Artifact untuk Web/API

In [14]:
# Simpan model final .keras
# Catatan: error _DictWrapper biasanya muncul saat export SavedModel pada beberapa versi TF/Keras.
# Karena kriteria hanya meminta .keras ATAU SavedModel, .keras dijadikan format utama yang siap produksi.

import shutil

# Dibuat ulang dengan input/output list agar lebih stabil saat disimpan.
model_to_export = tf.keras.Model(
    inputs=model.inputs,
    outputs=model.outputs,
    name=model.name,
)

keras_saved = False
savedmodel_saved = False

try:
    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
except AttributeError:
    pass

try:
    model_to_export.save(str(MODEL_PATH))
    keras_saved = True
    print("Model .keras tersimpan di:", MODEL_PATH)
except Exception as e:
    print("model.save .keras gagal:", e)


# Export SavedModel opsional.
# Tidak memakai tf.saved_model.save fallback karena pada environment tertentu memicu error:
# TypeError: this __dict__ descriptor does not support '_DictWrapper' objects
try:
    if SAVEDMODEL_DIR.exists():
        shutil.rmtree(SAVEDMODEL_DIR)

    if hasattr(model_to_export, "export"):
        model_to_export.export(str(SAVEDMODEL_DIR))
        savedmodel_saved = True
        print("SavedModel tersimpan di:", SAVEDMODEL_DIR)
    else:
        print("model.export tidak tersedia pada versi Keras/TensorFlow ini. SavedModel dilewati.")
except Exception as e:
    print("Export SavedModel dilewati karena gagal pada environment ini:", e)
    print("Model .keras tetap digunakan sebagai format produksi.")

if not keras_saved and not savedmodel_saved:
    raise RuntimeError("Model gagal disimpan. Cek versi TensorFlow/Keras dan custom layer.")


# Siapkan recommendation table agar inference.py tidak perlu membaca CSV lagi
recommendation_cols = [
    "title",
    "risk_label",
    "automation_risk_score",
    "reskilling_recommendation",
    "top_skills"
]

# Pastikan hanya kolom yang tersedia yang dipakai, agar tidak error jika nama kolom berbeda di CSV.
available_recommendation_cols = [col for col in recommendation_cols if col in df.columns]
missing_recommendation_cols = [col for col in recommendation_cols if col not in df.columns]
if missing_recommendation_cols:
    print("Kolom rekomendasi yang tidak ditemukan dan dilewati:", missing_recommendation_cols)


def _json_safe(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(k): _json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(v) for v in value]
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="ignore")
    return value


recommendation_records = []
for record in df[available_recommendation_cols].fillna("").head(5000).to_dict(orient="records"):
    recommendation_records.append({str(k): _json_safe(v) for k, v in record.items()})

# Pastikan semua artifact JSON serializable
artifacts = {
    "label_classes": [str(x) for x in label_classes],
    "numeric_cols": [str(x) for x in numeric_cols],
    "numeric_medians": {
        str(k): float(v) for k, v in numeric_medians.items()
    },
    "text_cols": [str(x) for x in TEXT_COLS],
    "target_label_col": str(TARGET_LABEL_COL),
    "target_score_col": str(TARGET_SCORE_COL),
    "model_format": {
        "keras_saved": bool(keras_saved),
        "keras_path": str(MODEL_PATH),
        "savedmodel_saved": bool(savedmodel_saved),
        "savedmodel_path": str(SAVEDMODEL_DIR),
        "primary_format": ".keras" if keras_saved else "SavedModel",
        "outputs": ["risk_label", "risk_score"],
    },
    "recommendation_records": recommendation_records,
}

with open(ARTIFACT_PATH, "w", encoding="utf-8") as f:
    json.dump(artifacts, f, ensure_ascii=False, indent=2)

metrics = {
    "test_accuracy": float(final_acc),
    "test_mae": float(final_mae),
    "target_accuracy": float(TARGET_ACC),
    "target_mae": float(TARGET_MAE),
    "passed_target": bool(final_acc >= TARGET_ACC and final_mae <= TARGET_MAE),
    "label_classes": [str(x) for x in label_classes],
}

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("Artifact tersimpan di:", ARTIFACT_PATH)
print("Metrics tersimpan di:", METRICS_PATH)


Model .keras tersimpan di: exports/worksafe_risk_model.keras
Export SavedModel dilewati karena gagal pada environment ini: this __dict__ descriptor does not support '_DictWrapper' objects
Model .keras tetap digunakan sebagai format produksi.
Artifact tersimpan di: exports/worksafe_artifacts.json
Metrics tersimpan di: exports/worksafe_metrics.json


## 15. Tes Inference Langsung dari Notebook

In [15]:
def build_job_text(title, description="", top_skills="", top_core_tasks=""):
    return (
        f"Job Title: {title}"
        f"Description: {description}"
        f"Top Skills: {top_skills}"
        f"Core Tasks: {top_core_tasks}"
    )


def predict_risk(title, description="", top_skills="", top_core_tasks="", numeric_features=None):
    numeric_features = numeric_features or {}

    job_text = build_job_text(title, description, top_skills, top_core_tasks)
    num_values = []
    for col in numeric_cols:
        num_values.append(float(numeric_features.get(col, numeric_medians.get(col, 0.0))))

    inputs = {
        "job_text": np.array([job_text], dtype=object),
        "numeric_features": np.array([num_values], dtype="float32"),
    }

    pred_label, pred_score = model.predict(inputs, verbose=0)
    label_idx = int(np.argmax(pred_label[0]))

    return {
        "risk_label": label_classes[label_idx],
        "confidence": float(pred_label[0][label_idx]),
        "automation_risk_score": float(pred_score[0][0]),
        "risk_percent": round(float(pred_score[0][0]) * 100, 2),
    }

sample = predict_risk(
    title="Warehouse Administration Staff",
    description="Responsible for data entry, inventory documentation, warehouse reports, and repetitive administrative tasks.",
    top_skills="data entry, inventory management, spreadsheet, reporting",
    top_core_tasks="record inventory movement, prepare reports, check warehouse documents",
)

sample

{'risk_label': 'Medium',
 'confidence': 0.6814903616905212,
 'automation_risk_score': 0.5895787477493286,
 'risk_percent': 58.96}

## Menjalankan TensorBoard

Setelah training, jalankan di terminal/notebook untuk melihat grafik loss, akurasi, dan MAE.

In [16]:
# Di notebook:
# %load_ext tensorboard
# %tensorboard --logdir logs/worksafe_nlp

# Di terminal:
# tensorboard --logdir logs/worksafe_nlp